## 45_setup_new_ws  (NEW workspace — Azure Databricks Premium + UC, via the Bastion jumphost)

Creates the landing zone for the migration (video frame 26):
  catalog practice_ml -> schema models -> volume staging_artifacts
Plus the three environment catalogs used by the DAB targets:
  practice_ml_dev | practice_ml_stg | practice_ml_prd
  (each with schema `practice` — the OLD schema name, kept IDENTICAL on both
   sides; 47_import_tables lifts the old schema + tables into them)
The volume replaces the video's ADLS "intermediate storage" — zero incremental
cost inside the Azure free-account credits.
After this, upload models_export.zip, jobs_export.zip AND tables_export.zip
into the volume via Catalog Explorer (from the jumphost browser session;
right-click the volume -> upload files).
NOTE: if CREATE CATALOG is denied (unlikely on Premium + UC), deploy with an
  override: databricks bundle deploy -t dev --var target_catalog=<existing_catalog>


In [ ]:


import re

def sql_id(x):
    assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", x), f"Invalid identifier: {x}"
    return x

CATALOG = "practice_ml"
SCHEMA = "models"
VOLUME = "staging_artifacts"
ENV_CATALOGS = ["practice_ml_dev", "practice_ml_stg", "practice_ml_prd"]

try:
    catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
except Exception:
    catalog = "spark_catalog"


In [ ]:


try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {sql_id(CATALOG)}")
    print(f"Catalog ready: {CATALOG}")
except Exception as e:
    print(f"Could not create catalog '{CATALOG}' ({str(e)[:120]}...)")
    print(f"Falling back to your default catalog: {catalog}")
    CATALOG = catalog

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {sql_id(CATALOG)}.{sql_id(SCHEMA)}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {sql_id(CATALOG)}.{sql_id(SCHEMA)}.{sql_id(VOLUME)}")


In [ ]:


# Environment catalogs for the dev -> stg -> prd bundle targets.
for env_catalog in ENV_CATALOGS:
    try:
        spark.sql(f"CREATE CATALOG IF NOT EXISTS {sql_id(env_catalog)}")
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {sql_id(env_catalog)}.practice")
        print(f"Environment catalog ready: {env_catalog}.practice (OLD schema name)")
    except Exception as e:
        print(f"Could not create {env_catalog} ({str(e)[:120]}...)")
        print(f"  -> Override at deploy time: --var target_catalog=<existing_catalog>")


In [ ]:


print("=== LANDING ZONE READY ===")
print(f"Catalog : {CATALOG}")
print(f"Schema  : {CATALOG}.{SCHEMA}")
print(f"Volume  : /Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
print(f"Models will be registered as: {CATALOG}.{SCHEMA}.<name>  (alias 'latest')")
print(f"Job env catalogs: {', '.join(ENV_CATALOGS)}  (schema 'practice' — the OLD schema name)")
print()
print("NEXT (UI): Catalog Explorer ->", CATALOG, "->", SCHEMA, "->", VOLUME)
print("  1. Upload models_export.zip   (from the old workspace)")
print("  2. Upload jobs_export.zip     (from the old workspace)")
print("  3. Upload tables_export.zip   (from the old workspace)")
print("Then run: 47_import_tables (tables/schema lift) -> 50_register_models")
print()
print("=== USE THESE WIDGET VALUES DOWNSTREAM (access via CATALOG widget OR CFG) ===")
print("  47_import_tables : TARGET_CATALOG=practice_ml_dev   OLD_SCHEMA=practice")
print("  65_inference     : CATALOG=practice_ml_dev          MODEL_CATALOG=practice_ml")
print("  Stage switch     : dev -> practice_ml_dev | stg -> practice_ml_stg | prd -> practice_ml_prd")
print("  (one widget word — or setup_config(environment=...) via configs/environments.yml —")
print("   switches the catalog; SCHEMA stays 'practice', no code change)")
